# P3 (ML_F) CHURN PREDICTION

### Importing Modules

In [60]:
# data manipulation & exploration
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# Automated EDA
!pip install sweetviz
import sweetviz as sv

# Classification models
from sklearn.linear_model import LogisticRegression, RidgeClassifier

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from scipy.stats import skew
from scipy.sparse import issparse
import time

# Evaluation metrics
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

In [61]:
train = pd.read_csv('Churn_train.csv')
test = pd.read_csv('Churn_test.csv')

In [62]:
# Drop the REF_NO column which is just an index
if 'customerID' in train.columns:
    train.drop(columns=['customerID'], inplace=True)

In [63]:
def add_family_strength(df):
    # Map Partner and Dependents to binary
    df['Partner_bin'] = df['Partner'].map({'Yes': 1, 'No': 0})
    df['Dependents_bin'] = df['Dependents'].map({'Yes': 1, 'No': 0})
    
    # Gender contributes 1 by default
    df['FamilyStrength'] = 1 + df['Partner_bin'] + df['Dependents_bin']
    
    # Optionally drop intermediate binary columns
    df.drop(columns=['Partner_bin', 'Dependents_bin'], inplace=True)
    
    return df

# Apply to both datasets
train = add_family_strength(train)
test = add_family_strength(test)

In [64]:
def add_num_phone_services(df):
    # Map PhoneService: Y → 1, N → 0
    df['PhoneService_bin'] = df['PhoneService'].map({'Yes': 1, 'No': 0})
    
    # Map MultiplePhoneLines: Y → 1, N & No service → 0
    df['MultipleLines_bin'] = df['MultipleLines'].map({'Yes': 1, 'No': 0, 'No phone service': 0})
    
    # Sum both binary features
    df['NumPhoneServices'] = df['PhoneService_bin'] + df['MultipleLines_bin']
    
    # Optionally drop intermediate binary columns
    df.drop(columns=['PhoneService_bin', 'MultipleLines_bin'], inplace=True)
    
    return df

# Apply to both datasets
train = add_num_phone_services(train)
test = add_num_phone_services(test)

In [65]:
def add_num_internet_services(df):
    # Map InternetService: Fiber optic & DSL → 1, No → 0
    df['InternetService_bin'] = df['InternetService'].map({'Fiber optic': 1, 'DSL': 1, 'No': 0})
    
    # Map other service features: Y → 1, N & No service → 0
    service_cols = [
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]
    
    for col in service_cols:
        df[col + '_bin'] = df[col].map({'Yes': 1, 'No': 0, 'No internet service': 0})
    
    # Sum all binary service columns
    df['NumInternetServices'] = df['InternetService_bin'] + df[[col + '_bin' for col in service_cols]].sum(axis=1)
    
    # Optionally drop intermediate binary columns
    df.drop(columns=['InternetService_bin'] + [col + '_bin' for col in service_cols], inplace=True)
    
    return df

# Apply to both datasets
train = add_num_internet_services(train)
test = add_num_internet_services(test)

In [66]:
train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce')
test['TotalCharges'] = pd.to_numeric(test['TotalCharges'], errors='coerce')
train['TotalCharges'].fillna(train['MonthlyCharges'] * train['tenure'], inplace=True)
test['TotalCharges'].fillna(test['MonthlyCharges'] * test['tenure'], inplace=True)

C:\Users\prpatel1\AppData\Local\Temp\ipykernel_3976\515123816.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['TotalCharges'].fillna(train['MonthlyCharges'] * train['tenure'], inplace=True)
C:\Users\prpatel1\AppData\Local\Temp\ipykernel_3976\515123816.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting va

In [67]:
def add_avg_monthly_charges(df):
    # Avoid division by zero or missing tenure
    df['AvgMonthlyCharges'] = df.apply(
        lambda row: row['TotalCharges'] / row['tenure'] if row['tenure'] > 0 else 0,
        axis=1
    )
    return df

# Apply to both datasets
train = add_avg_monthly_charges(train)
test = add_avg_monthly_charges(test)

In [68]:
# Convert Dependent to Numeric
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0})

### Prepare Target

In [69]:
test_ref = test['customerID']

In [70]:
y = train['Churn']
X = train.drop(['Churn'], axis = 1)
test = test.drop(['customerID'], axis = 1)

### Feature Engineering

In [71]:
nominal_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
               'PaperlessBilling', 'PaymentMethod']

num_cols = X.select_dtypes(include = ['float64', 'int64']).columns.tolist()

onehot_encoder = OneHotEncoder(handle_unknown='ignore')
scaler = StandardScaler()

preprocessor = ColumnTransformer([
    ('onehot', onehot_encoder, nominal_cols),
    ('scale', scaler, num_cols)
])

X_processed = preprocessor.fit_transform(X)
test_processed = preprocessor.transform(test)

if issparse(X_processed):
    X_processed = X_processed.toarray()
    test_processed = test_processed.toarray()

In [72]:
# Access the fitted OneHotEncoder inside the ColumnTransformer
fitted_onehot_encoder = preprocessor.named_transformers_['onehot']

# Accessing one-hot encoded categories
for col, cats in zip(nominal_cols, fitted_onehot_encoder.categories_):
    print(f'\nFeature: {col}')
    for i, cat in enumerate(cats):
        print(f' {cat} ---> Column {i}')


Feature: gender
 Female ---> Column 0
 Male ---> Column 1

Feature: Partner
 No ---> Column 0
 Yes ---> Column 1

Feature: Dependents
 No ---> Column 0
 Yes ---> Column 1

Feature: PhoneService
 No ---> Column 0
 Yes ---> Column 1

Feature: MultipleLines
 No ---> Column 0
 No phone service ---> Column 1
 Yes ---> Column 2

Feature: InternetService
 DSL ---> Column 0
 Fiber optic ---> Column 1
 No ---> Column 2

Feature: OnlineSecurity
 No ---> Column 0
 No internet service ---> Column 1
 Yes ---> Column 2

Feature: OnlineBackup
 No ---> Column 0
 No internet service ---> Column 1
 Yes ---> Column 2

Feature: DeviceProtection
 No ---> Column 0
 No internet service ---> Column 1
 Yes ---> Column 2

Feature: TechSupport
 No ---> Column 0
 No internet service ---> Column 1
 Yes ---> Column 2

Feature: StreamingTV
 No ---> Column 0
 No internet service ---> Column 1
 Yes ---> Column 2

Feature: StreamingMovies
 No ---> Column 0
 No internet service ---> Column 1
 Yes ---> Column 2

Feature

In [73]:
#X & test processing check

# Get feature names from each transformer
onehot_feature_names = fitted_onehot_encoder.get_feature_names_out(nominal_cols)
scaled_feature_names = num_cols  # StandardScaler also keeps original names

# Combine all feature names
all_feature_names = list(onehot_feature_names) + list(scaled_feature_names)

# Convert to DataFrame with column names
X_df = pd.DataFrame(X_processed, columns=all_feature_names)
test_df = pd.DataFrame(test_processed, columns=all_feature_names)

In [74]:
# Set display option to show all columns
pd.set_option('display.max_columns', None)

X_df.head()

,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,FamilyStrength,NumPhoneServices,NumInternetServices,AvgMonthlyCharges
0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,-0.439475,-0.825884,-1.497530,-0.890947,1.503255,-0.507510,-1.334413,-1.427580
1,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,-0.439475,0.395961,0.302996,0.389693,-0.951626,-0.507510,1.506352,0.346954
2,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,-0.439475,1.577078,0.012320,1.060945,0.275815,1.046464,0.559430,0.039393
3,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.439475,1.577078,0.686687,1.775397,1.503255,1.046464,1.506352,0.792658
4,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.439475,-0.092777,0.186726,-0.102671,-0.951626,-0.507510,1.032891,0.112664


### Validation Split

In [75]:
X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, stratify = y, random_state = 42)

### HPT

In [76]:
# Define parameter grid for Logistic Regression
param_grid_elastic = {
    'C': [0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.9],  # ElasticNet mixing parameter
    'penalty': ['elasticnet'],
    'solver': ['saga'],
    'max_iter': [1000]
}

# Initialize base model
base_lr = LogisticRegression()

# Set up GridSearchCV
grid_search_lr = GridSearchCV(
    estimator=base_lr,
    param_grid=param_grid_elastic,
    scoring='accuracy',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=2
)

# Fit GridSearchCV
print("\nStarting GridSearchCV for Logistic Regression...")
grid_search_lr.fit(X_train, y_train)

# Get best model
best_lr = grid_search_lr.best_estimator_
print(f"Best parameters: {grid_search_lr.best_params_}")
print(f"Best CV accuracy: {grid_search_lr.best_score_:.4f}")

# Replace model in your loop
models = {'lr': best_lr}


Starting GridSearchCV for Logistic Regression...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best parameters: {'C': 1, 'l1_ratio': 0.1, 'max_iter': 1000, 'penalty': 'elasticnet', 'solver': 'saga'}
Best CV accuracy: 0.8039


### Model Training

In [77]:
start_all = time.time()

In [78]:
for name, model in models.items():
    print(f'\n Training model: {name.upper()}')
    t0 = time.time()
    model.fit(X_train, y_train)
    print(f'Trained in {round(time.time() - t0, 2)} seconds')
    
    # Evaluate on validation set
    t0 = time.time()
    val_preds = model.predict(X_val)
    
    acc = accuracy_score(y_val, val_preds)
    precision = precision_score(y_val, val_preds)
    recall = recall_score(y_val, val_preds)
    f1 = f1_score(y_val, val_preds)
    f1_macro = f1_score(y_val, val_preds, average = 'macro')
    print(f"Classification report: \n{classification_report(y_val, val_preds, target_names=['LNI', 'HNI'])}")
    print(f'\nMetrics:')
    print(f'- Accuracy: {round(acc, 4)}')
    print(f'- Precision: {round(precision, 4)}')
    print(f'- Recall: {round(recall, 4)}')
    print(f'- F1 Score: {round(f1, 4)}')
    print(f'- F1 Macro: {round(f1_macro, 4)}')
    print(f'Validation done in {round(time.time() - t0, 2)} seconds')
    
    # Plot confusion matrix
    cm = confusion_matrix(y_val, val_preds)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['LNI', 'HNI'], yticklabels=['LNI', 'HNI'])
    plt.title(f'Confusion Matrix: {name.upper()}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{name}.png')
    plt.close()
    
    # Predict on real test set
    t0 = time.time()
    test_preds = model.predict(test_processed)
    test_preds_final = pd.Series(test_preds).map({1: "Yes", 0: "No"}) # Reverse mapping for submission
    
    submission = pd.DataFrame({
        'REF_NO': test_ref,
        'Revenue_Grid': test_preds_final
    })
    
    submission.to_csv(f'submission_{name}.csv', index = False)
    print(f'Submission saved as submission_{name}.csv in {round(time.time() - t0, 2)} seconds')


 Training model: LR
Trained in 0.93 seconds
Classification report: 
              precision    recall  f1-score   support

         LNI       0.85      0.89      0.87       823
         HNI       0.66      0.58      0.62       304

    accuracy                           0.81      1127
   macro avg       0.76      0.74      0.74      1127
weighted avg       0.80      0.81      0.80      1127


Metrics:
- Accuracy: 0.8075
- Precision: 0.6642
- Recall: 0.5789
- F1 Score: 0.6186
- F1 Macro: 0.7449
Validation done in 0.01 seconds
Submission saved as submission_lr.csv in 0.0 seconds


### Threshold Tuning Check

In [79]:
# Get predicted probabilities for the positive class
y_proba = best_lr.predict_proba(X_val)[:, 1]

# Define thresholds to evaluate
thresholds = np.arange(0.1, 0.91, 0.01)
accuracies = []

# Evaluate accuracy at each threshold
for threshold in thresholds:
    y_pred = (y_proba >= threshold).astype(int)
    acc = accuracy_score(y_val, y_pred)
    accuracies.append(acc)

# Find the best threshold
best_threshold = thresholds[np.argmax(accuracies)]
best_accuracy = max(accuracies)

# Plot accuracy vs threshold
plt.figure(figsize=(8, 5))
plt.plot(thresholds, accuracies, marker='o')
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best Threshold = {best_threshold:.2f}')
plt.title('Accuracy vs. Threshold for Logistic Regression')
plt.xlabel('Threshold')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('threshold_tuning_logreg.png')
plt.close()

# Print best threshold and accuracy
print(f"Best Threshold: {best_threshold:.2f}")
print(f"Best Accuracy: {best_accuracy:.4f}")

Best Threshold: 0.50
Best Accuracy: 0.8075
